In [1]:
import pandas as pd
import numpy as np
import random
#import faker
#namegen = faker.Faker()
from Handler import *
# plotnine

pygame 2.6.1 (SDL 2.28.4, Python 3.13.12)
Hello from the pygame community. https://www.pygame.org/contribute.html
0


c:\Users\leojo\OneDrive\Documents\GitHub\AWH_Curling3\PlayerMovement.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  FADF = pd.concat([ROOKIEDF, EXPIREDF]).sort_values(['Rating', 'Age'], ascending=[False, True])
c:\Users\leojo\OneDrive\Documents\GitHub\AWH_Curling3\PlayerMovement.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  NEEDSDF = TEAMSDF.merge(RosterDF.groupby(['Team'], sort = False).agg(Spent = ('AAV', 'sum'), NR = ('AAV', 'count')), how = 'left', left_index=True, right_index=True).fillna(0)
c:\Users\leojo\

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19


In [38]:
class HANDLER:
    def __init__(self):
        self.YEAR = 1
        self.slate = 1
        self.RESULTS = pd.DataFrame(columns = ['YEAR', 'COMPETITION', 'STAGE', 'Home', 'HomeScore', 'AwayScore', 'Away', 'ExtraEnd'])
        self.WINNERS = pd.DataFrame(columns = ['YEAR', 'COMPETITION', 'WINNER'])
        self.WorldCup = None
        self.CWC = None
        self.WCQ = {
            'Europe': None,
            'Asia': None,
            'Americas': None,
            'MEA': None
        }
        self.CL = {
            'Europe': None,
            'Asia': None,
            'Americas': None,
            'MEA': None
        }
        self.LEAGUES = {cnty: LEAGUE(cnty, TEAMSDF[TEAMSDF.Country == cnty].index) for cnty in COUNTRYDF.index}

    def displayNextSlate(self):
        if self.slate % 2 == 1: # League Slate
            for cnty in self.LEAGUES.keys():
                print(f'Slate for {cnty}')
                self.LEAGUES[cnty].displayNextSlate()

    def playNextSlate(self):
        if self.slate % 2 == 1: # League Slate
            for cnty in self.LEAGUES.keys():
                self.LEAGUES[cnty].playNextSlate(self)
        self.slate += 1
GLOBALHANDLER = HANDLER()

In [40]:
def game_LoRes(Home, Away):
    HomeRating = Home.getRating()
    AwayRating = Away.getRating()
    diff = .21*(HomeRating - AwayRating)
    spread = math.floor(np.random.normal(diff, 3))
    loserScore = round(12*np.random.beta(9, 4))
    if spread < 0:
        return [Home, loserScore, loserScore-spread, Away, False]
    elif spread > 0:
        return [Home, loserScore+spread, loserScore, Away, False]
    else:
        return [Home, loserScore+1, loserScore, Away, True]

In [172]:
class CL_Europe:
    def __init__(self, handle):
        self.AutoSwiss = []
        self.Qualifiers = []
        self.handle = handle
        self.slate = 1
        self.winner = None
        self.QualfyingTs = None
        self.Swiss = None
        self.Playoffs = None
        TIERA = list(COUNTRYDF[(COUNTRYDF.Region == 'Europe') & (COUNTRYDF.TIER == 'TIERA')].index)
        TIERB = list(COUNTRYDF[(COUNTRYDF.Region == 'Europe') & (COUNTRYDF.TIER == 'TIERB')].index)
        TIERC = list(COUNTRYDF[(COUNTRYDF.Region == 'Europe') & (COUNTRYDF.TIER == 'TIERC')].index)
        TIERD = list(COUNTRYDF[(COUNTRYDF.Region == 'Europe') & (COUNTRYDF.TIER == 'TIERD')].index)
        TIERE = list(COUNTRYDF[(COUNTRYDF.Region == 'Europe') & (COUNTRYDF.TIER == 'TIERE')].index)
        random.shuffle(TIERD) # Needed cuz one will get bye
        for cnty in TIERA:
            self.AutoSwiss.append(GLOBALHANDLER.LEAGUES[cnty].tWinner)
            self.AutoSwiss.append(GLOBALHANDLER.LEAGUES[cnty].popqual())
            self.AutoSwiss.append(GLOBALHANDLER.LEAGUES[cnty].popqual())
        for cnty in TIERB:
            self.AutoSwiss.append(GLOBALHANDLER.LEAGUES[cnty].tWinner)
            self.AutoSwiss.append(GLOBALHANDLER.LEAGUES[cnty].popqual())
        for cnty in TIERC:
            self.AutoSwiss.append(GLOBALHANDLER.LEAGUES[cnty].tWinner)
        for cnty in TIERD+TIERE:
            self.Qualifiers.append(GLOBALHANDLER.LEAGUES[cnty].tWinner)
        for cnty in TIERA+TIERC+TIERD:
            self.Qualifiers.append(GLOBALHANDLER.LEAGUES[cnty].popqual())
        self.Qualifiers.append(None)
        self.QualifyingTs = [Bracket(f'Europe CL Quals Path {num+1}', [self.Qualifiers[num], self.Qualifiers[13-num], self.Qualifiers[14+num], self.Qualifiers[27-num]]) for num in range(7)]

    def playNextSlate(self):
        if self.slate <= 2:
            for brkt in self.QualifyingTs:
                brkt.playNextSlate(self.handle)
            if self.slate == 2:
                self.Swiss = Swiss('Europe CL Swiss', self.AutoSwiss+[brkt.winner for brkt in self.QualifyingTs])
        elif self.slate <= 7:
            self.Swiss.playNextSlate(self.handle)
            if self.slate == 7:
                advanced = self.Swiss.getAdvanced()
                random.shuffle(advanced)
                self.Playoffs = Bracket('Europe CL Knockouts', advanced)
        elif self.slate <= 11:
            self.Playoffs.playNextSlate(self.handle)
            if self.slate == 11:
                self.winner = self.Playoffs.winner
        else:
            print('Season Over')
        self.slate += 1

    def displayNextSlate(self):
        if self.slate <= 2:
            for brkt in self.QualifyingTs:
                brkt.displayNextSlate()
        elif self.slate <= 7:
            self.Swiss.displayNextSlate()
        elif self.slate <= 11:
            self.Playoffs.displayNextSlate()
        else:
            print('Season Over')

    def teamIn(self, tm):
        return tm in self.AutoSwiss+self.Qualifiers
    
class CL_Asia:
    def __init__(self, handle):
        self.handle = handle
        self.AutoDE = []
        self.Qualifiers = []
        self.slate = 1
        self.winner = None
        self.QualfyingTs = None
        self.DE = None
        TIERA = list(COUNTRYDF[(COUNTRYDF.Region == 'Asia') & (COUNTRYDF.TIER == 'TIERA')].index)
        TIERB = list(COUNTRYDF[(COUNTRYDF.Region == 'Asia') & (COUNTRYDF.TIER == 'TIERB')].index)
        TIERC = list(COUNTRYDF[(COUNTRYDF.Region == 'Asia') & (COUNTRYDF.TIER == 'TIERC')].index)
        TIERD = list(COUNTRYDF[(COUNTRYDF.Region == 'Asia') & (COUNTRYDF.TIER == 'TIERD')].index)
        TIERE = list(COUNTRYDF[(COUNTRYDF.Region == 'Asia') & (COUNTRYDF.TIER == 'TIERE')].index)
        for cnty in TIERA+TIERB+TIERC+TIERD+TIERE:
            self.AutoDE.append(GLOBALHANDLER.LEAGUES[cnty].tWinner)
        for cnty in TIERA+TIERB+TIERC:
            self.AutoDE.append(GLOBALHANDLER.LEAGUES[cnty].popqual())
        for cnty in TIERA+TIERB+TIERD:
            self.Qualifiers.append(GLOBALHANDLER.LEAGUES[cnty].popqual())
        for cnty in TIERA:
            self.Qualifiers.append(GLOBALHANDLER.LEAGUES[cnty].popqual())
        self.QualifyingTs = [Bracket(f'Asia CL Quals Path {num+1}', [self.Qualifiers[num], self.Qualifiers[15-num]]) for num in range(8)]

    def playNextSlate(self):
        if self.slate <= 1:
            for brkt in self.QualifyingTs:
                brkt.playNextSlate(self.handle)
            if self.slate == 1:
                self.DE = DubElim32('Asia CL', self.AutoDE+[brkt.winner for brkt in self.QualifyingTs])
        elif self.slate <= 11:
            self.DE.playNextSlate(self.handle)
            if self.slate == 11:
                self.winner = self.DE.winner
        else:
            print('Season Over')
        self.slate += 1

    def displayNextSlate(self):
        if self.slate <= 1:
            for brkt in self.QualifyingTs:
                brkt.displayNextSlate()
        elif self.slate <= 11:
            self.DE.displayNextSlate()
        else:
            print('Season Over')

    def teamIn(self, tm):
        return tm in self.AutoDE+self.Qualifiers

class CL_Americas:
    def __init__(self, handle):
        self.handle = handle
        self.ToStage2 = []
        self.ToStage1 = []
        self.slate = 1
        self.winner = None
        self.Stage1s = None
        self.Stage2s = None
        self.Stage3s = None
        self.Finals = None
        TIERA = list(COUNTRYDF[(COUNTRYDF.Region == 'Americas') & (COUNTRYDF.TIER == 'TIERA')].index)
        TIERB = list(COUNTRYDF[(COUNTRYDF.Region == 'Americas') & (COUNTRYDF.TIER == 'TIERB')].index)
        TIERC = list(COUNTRYDF[(COUNTRYDF.Region == 'Americas') & (COUNTRYDF.TIER == 'TIERC')].index)
        TIERD = list(COUNTRYDF[(COUNTRYDF.Region == 'Americas') & (COUNTRYDF.TIER == 'TIERD')].index)
        TIERE = list(COUNTRYDF[(COUNTRYDF.Region == 'Americas') & (COUNTRYDF.TIER == 'TIERE')].index)
        for cnty in TIERA+TIERB+TIERC:
            self.ToStage2.append(GLOBALHANDLER.LEAGUES[cnty].tWinner)
        for cnty in TIERA:
            self.ToStage2.append(GLOBALHANDLER.LEAGUES[cnty].popqual())
        for cnty in TIERD+TIERE:
            self.ToStage1.append(GLOBALHANDLER.LEAGUES[cnty].tWinner)
        for cnty in TIERA+TIERB+TIERC+TIERD:
            self.ToStage1.append(GLOBALHANDLER.LEAGUES[cnty].popqual())
        for cnty in TIERA+TIERB+TIERC:
            self.ToStage1.append(GLOBALHANDLER.LEAGUES[cnty].popqual())
        for cnty in TIERA+TIERB:
            self.ToStage1.append(GLOBALHANDLER.LEAGUES[cnty].popqual())
        random.shuffle(self.ToStage1)
        self.Stage1s = [DE4(f'Americas CL Stage 1 Path {num//4+1}', self.ToStage1[num:(num+4)]) for num in range(0, 20, 4)]

    def playNextSlate(self):
        if self.slate <= 3:
            for brkt in self.Stage1s:
                brkt.playNextSlate(self.handle)
            if self.slate == 3:
                s2t = self.ToStage2
                for brkt in self.Stage1s:
                    s2t = s2t + brkt.getAdvanced()
                random.shuffle(s2t)
                self.Stage2s = [DE4(f'Americas CL Stage 2 Path {num//4+1}', s2t[num:(num+4)]) for num in range(0, 16, 4)]
        elif self.slate <= 6:
            for brkt in self.Stage2s:
                brkt.playNextSlate(self.handle)
            if self.slate == 6:
                s3t = []
                for brkt in self.Stage2s:
                    s3t = s3t + brkt.getAdvanced()
                random.shuffle(s3t)
                self.Stage3s = [DE4(f'Americas CL Stage 3 Path {num//4+1}', s3t[num:(num+4)]) for num in range(0, 8, 4)]
        elif self.slate <= 9:
            for brkt in self.Stage3s:
                brkt.playNextSlate(self.handle)
            if self.slate == 9:
                pt = []
                for brkt in self.Stage3s:
                    pt = pt + brkt.getAdvanced()
                random.shuffle(pt)
                self.Finals = Bracket(f'Americas CL Finals', pt)
        elif self.slate <= 11:
            self.Finals.playNextSlate(self.handle)
            if self.slate == 11:
                self.winner = self.Finals.winner
        else:
            print('Season Over')
        self.slate += 1

    def displayNextSlate(self):
        if self.slate <= 3:
            for brkt in self.Stage1s:
                brkt.displayNextSlate()
        elif self.slate <= 6:
            for brkt in self.Stage2s:
                brkt.displayNextSlate()
        elif self.slate <= 9:
            for brkt in self.Stage3s:
                brkt.displayNextSlate()
        elif self.slate <= 11:
            self.Finals.displayNextSlate()
        else:
            print('Season Over')

    def teamIn(self, tm):
        return tm in self.ToStage1+self.ToStage2




    

In [42]:
for _ in range(45):
    GLOBALHANDLER.playNextSlate()

In [173]:
CLAs = CL_Americas(GLOBALHANDLER)

In [196]:
CLAs.displayNextSlate()

Season Over


In [198]:
CLAs.winner.name

'Sao Paulo'

In [97]:
TESTDE = DubElim32('TESTME', list(TEAMSDF.index[:32]))

In [118]:
TESTDE.displayNextSlate()

In [117]:
TESTDE.playNextSlate(GLOBALHANDLER)

In [120]:
GLOBALHANDLER.WINNERS

,YEAR,COMPETITION,WINNER
0,1,Algeria League Play,Oran
1,1,Angola League Play,Luanda
2,1,Argentina League Play,Rosario
3,1,Australia League Play,Sunshine Coast
4,1,Austria League Play,Vienna
...,...,...,...
140,1,Europe CL Quals Path 5,Bucharest
141,1,Europe CL Quals Path 6,Copenhagen
142,1,Europe CL Quals Path 7,Gaziantep
143,1,Europe CL Knockouts,Zurich
